# Market handles

The market snapshot through `tq.MarketEnvironment`: rate curves, equities (spot, repo, dividends), vol surfaces.
Docs: *Architecture* (Market data), *Models and Pricing* (Curves), *Exercises* (`rates.tex` Curves, `equity.tex`).

In [ ]:
import os
from pathlib import Path
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"

REPO = Path.cwd().resolve().parents[1]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorquant as tq

## Market
Latest snapshot in `marketdata/<yyyymmdd>/<HHMM>/`. Keys `RISKFACTOR:CCY:TICKER:TYPE`; each market object holds its
risk factors as float64 `tf.Variable`s.

In [ ]:
mkt = tq.MarketEnvironment.from_folder(REPO / "marketdata")   # latest snapshot
tq.Settings.evaluation_date = mkt.evaluation_date

factors = mkt.risk_factors()                                   # {key: [tf.Variable, ...]}
count = pd.Series([f"{k.split(':')[0]}:{k.split(':')[3]}" for k in factors]).value_counts()
print(f"snapshot {mkt.evaluation_date} / {mkt.calibration_set}")
print(count.to_string())

## Rate curves
One OIS curve per currency (discounting) + EUR 6M (Euribor projection). Zero rates (Act/365, continuous compounding) on
the pillars, linear in between, flat outside: $P(0,T) = e^{-r(T)\,T}$.

In [ ]:
curves = {k: mkt.get_ir_curve(tq.Currency[k.split(":")[1]], ticker=k.split(":")[2])
          for k in factors if k.startswith("IR:") and k.endswith(":SPOT")}       # (VOL / SWVOL are rate vols)

pd.DataFrame([{"key": k, "pillars": len(c.pillars), "first": c.dates[0], "last": c.dates[-1],
               "daycount": c.daycounter_convention.name, "interp": c.interpolation_type,
               "r(1Y) %": 100 * float(c.interp.interpolate(1.0)), "r(10Y) %": 100 * float(c.interp.interpolate(10.0))}
              for k, c in curves.items()]).round(3)

Zero rate, discount factor and forwards. Instantaneous: $f(0,T) = -\partial_T \ln P(0,T) = r(T) + T\,r'(T)$;
simple over $\tau$ = 6M: $F(0;T,T+\tau) = \frac{1}{\tau}\Big(\frac{P(0,T)}{P(0,T+\tau)} - 1\Big)$. Markers: the pillars.

In [ ]:
t = np.linspace(0.05, 30.0, 600)
tau = 0.5
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for key, c in curves.items():
    label = key.split(":")[2]
    df = np.array([float(c.discount(x)) for x in t])
    line, = axes[0, 0].plot(t, -np.log(df) / t * 100, label=label)
    axes[0, 0].plot(c.pillars, [100 * float(r) for r in c.rates], "o", ms=4, color=line.get_color())
    axes[0, 1].plot(t, df, label=label)
    axes[1, 0].plot(t, [100 * float(c.inst_fwd(x)) for x in t], label=label)
    axes[1, 1].plot(t, [100 * float(c.forward_rate(x, x + tau)) for x in t], label=label)
titles = ["Zero rate r(T), cont. comp. (%)", "Discount factor P(0,T)", "Instantaneous forward f(0,T) (%)",
          "Simple 6M forward F(0;T,T+6M) (%)"]
for ax, title in zip(axes.ravel(), titles):
    ax.set_title(title); ax.set_xlabel("T (years)"); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout()

Linear zero rates: $r'(T)$ jumps at each pillar, so does $f(0,T)$ (saw-tooth above; at a pillar `inst_fwd` gives the right
value, the forward of the next segment); the 6M forward averages it over the period and is continuous.

## Two curves: EUR 6M vs ESTR
ESTR discounts, the 6M curve projects the Euribor fixings: their 6M forwards differ by the tenor basis.

In [ ]:
estr, eur6m = curves["IR:EUR:ESTR:SPOT"], curves["IR:EUR:6M:SPOT"]
t6 = np.linspace(0.0, 29.5, 300)
basis = [1e4 * (float(eur6m.forward_rate(x, x + 0.5)) - float(estr.forward_rate(x, x + 0.5))) for x in t6]
plt.figure(figsize=(7, 3.5))
plt.plot(t6, basis); plt.title("6M forward: EUR 6M - ESTR (bp)"); plt.xlabel("T (years)"); plt.grid(alpha=0.3)

## Equities
`mkt.get_equity(ticker)`: a view on spot, repo, dividends, vol surface and discount curve. Dividend model from the data:
`DIV` present → discrete (escrowed), otherwise `DIVYIELD` continuous. Forward and dividends in detail: notebooks `dividends` and `equity_underlying`.

In [ ]:
TICKERS = ["TICKER1", "TICKER31", "TICKER41"]   # one per currency (EUR, GBP, USD)

rows = []
for ticker in TICKERS:
    eq = mkt.get_equity(ticker)
    spot = float(eq.spot)
    rows.append({"ticker": ticker, "ccy": eq.currency.name, "spot": spot, "repo %": 100 * float(eq.repo),
                 "dividends": eq.dividend_model, "n div": len(eq.dividends.amounts) if eq.dividends else 0,
                 "PV div 1Y / S %": 100 * float(eq.pv_dividends(1.0)) / spot,
                 "F(1Y) / S": float(eq.forward(1.0)) / spot, "curve": eq.discount_curve.name})
pd.DataFrame(rows).round(4)

In [ ]:
eq = mkt.get_equity("TICKER1")
pd.DataFrame({"ex date": eq.dividends.ex_dates, "amount": [float(a) for a in eq.dividends.amounts]}).head(6)

## Volatility surfaces
Black vols on (maturity × absolute strike) nodes: `volatility_matrix` is one variable (vega per node); `volatility(strike,
tenor)` is bilinear, flat outside the grid.

In [ ]:
fig, axes = plt.subplots(1, len(TICKERS), figsize=(5 * len(TICKERS), 4))
for ax, ticker in zip(axes, TICKERS):
    eq = mkt.get_equity(ticker)
    vs, spot = eq.vol_surface, float(eq.spot)
    K = np.array(vs.strike)
    for i, T in enumerate(vs.maturity):
        ax.plot(K / spot, 100 * vs.volatility_matrix.numpy()[i], "o-", ms=3, label=f"T={T:.2f}")
    ax.set_title(f"{ticker} ({eq.currency.name})"); ax.set_xlabel("K / spot"); ax.set_ylabel("Black vol (%)")
    ax.grid(alpha=0.3); ax.legend(fontsize=7, ncol=2)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(7, 3.5))
for ticker in TICKERS:
    eq = mkt.get_equity(ticker)
    vs, spot = eq.vol_surface, float(eq.spot)
    T = np.linspace(vs.maturity[0], vs.maturity[-1], 200)
    line, = plt.plot(T, [100 * float(vs.volatility(strike=spot, tenor=x)) for x in T], label=ticker)
    plt.plot(vs.maturity, [100 * float(vs.volatility(strike=spot, tenor=x)) for x in vs.maturity], "o", ms=3,
             color=line.get_color())
plt.title("Vol at K = spot vs maturity (%)"); plt.xlabel("T (years)"); plt.grid(alpha=0.3); plt.legend();

## Exercises
1. Check $f(0,T) \approx r(T) + T\,r'(T)$ with finite differences away from a pillar, and measure the jump of $f$ at a pillar.
2. Show that $\tau\,F(0;T,T+\tau) = e^{\int_T^{T+\tau} f(0,u)\,du} - 1$ (integrate `inst_fwd` numerically).
3. Bump one node of the TICKER1 `volatility_matrix` (`assign`) and find which `volatility(strike, tenor)` values move.